In [12]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import xmltodict, json
import ast
import numbers
import shlex # package to construct the git command to subprocess format
import subprocess 
import os
%matplotlib inline

In [13]:
ReleasedWheat = 'C:\GitHubRepos\ApsimX\Models\Resources\Wheat.json'
MasterFile = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.apsimx'
PrototypeFile = 'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatPrototype.apsimx'
ImplementedFile = 'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\WheatSL.apsimx'
VariableRenamesFile = 'C:\GitHubRepos\ApsimX\Prototypes\WheatSimpleLeaf\SimpleLeafImplementation\VariableRenames.xlsx'  

In [14]:
def findModel(Parent,modelPath):
    PathElements = modelPath.split('.')
    return findModelFromElements(Parent,PathElements)

def findModelFromElements(Parent,PathElements):
    for pe in PathElements:
        Parent = findNextChild(Parent,pe)
    return Parent

def findNextChild(Parent,ChildName):
    if len(Parent['Children']) >0:
        for child in range(len(Parent['Children'])):
            if Parent['Children'][child]['Name'] == ChildName:
                return Parent['Children'][child]
    else:
        return Parent[ChildName]

def replaceModel(Parent,modelPath,New):
    PathElements = modelPath.split('.')
    try:
        test = findModelFromElements(Parent,PathElements[:-1])[PathElements[-1]]
        findModelFromElements(Parent,PathElements[:-1])[PathElements[-1]] = New
    except:
        try:
            pos = 0
            for kid in findModelFromElements(Parent,PathElements[:-1])['Children']:
                if kid['Name'] == PathElements[-1]:
                    findModelFromElements(Parent,PathElements[:-1])['Children'][pos] = New
                    break
                pos +=1
        except:   
            print('Could not find parent node of model to over write for ' + modelPath)
            raise
            
def addModel(Parent,modelPath,New):
    PathElements = modelPath.split('.')
    Parent = findModelFromElements(Parent,PathElements)
    if Parent == None:
        print('Could not find parent model ' + modelPath + ' to Add new model to.  Dont include the name of the new models name in the path')
    if isinstance(New,dict):
        NewDict = New
    else:
        NewDict = json.loads(New)
    Parent['Children'].append(NewDict)
    
def renameModel(Parent,modelPath,NewName):
    PathElements = modelPath.split('.')
    Parent = findModel(Parent,PathElements)
    Parent['Name'] = NewName
    
def renameModelofType(Parent,modelName,modelType,NewName):
    for c in Parent['Children']:
        if (c['Name'] == modelName) and (c['$type'] == modelType):
            c['Name'] = NewName
        renameModelofType(c,modelName,modelType,NewName)
            
def removeModel(Parent,modelName,modelType):
    pos = 0
    for c in Parent['Children']:
        if (c['Name'] == modelName) and (c['$type'] == modelType):
            del Parent['Children'][pos]
            found = True
            break
        else:
            removeModel(c,modelName,modelType)
        pos += 1

In [15]:
# command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout upstream/master C:/GitHubRepos/ApsimX/Tests/Validation/Wheat/Wheat.apsimx" 
# #command= "git --git-dir=C:/GitHubRepos/ApsimX/.git --work-tree=C:/GitHubRepos/ApsimX checkout C:/GitHubRepos/ApsimX/Models/Resources/Wheat.json" 
# comm=shlex.split(command) # This will convert the command into list format
# subprocess.run(comm, shell=True) # Run the git command

In [16]:
## Read wheat test file into json object
with open(MasterFile,'r') as MasterJSON:
    Master = json.load(MasterJSON)
    MasterJSON.close()
    ## read prototype wheat file into json object
with open(PrototypeFile,'r') as PrototypeJSON:
    Prototype = json.load(PrototypeJSON)
    PrototypeJSON.close()

In [17]:
## Read released wheat model so we can bring its cultivar parameters across
with open(ReleasedWheat,'r') as ReleasedJSON:
    Released = json.load(ReleasedJSON)
    ReleasedJSON.close()

In [18]:
#Copy prototype wheat model out of replacements and put it in replacements in test file
NewModel =  findModel(Prototype,'Replacements.Wheat')
addModel(Master,'Replacements',NewModel)
#Put updated leaf size calculation script into test file
NewModel =  findModel(Prototype,'Replacements.OutputMaxLeafSize')
replaceModel(Master,'Replacements.MaxLeafSize',NewModel)
#Put updated leaf size report into test file
NewModel =  findModel(Prototype,'Replacements.ReportMaxLeafSize')
replaceModel(Master,'Replacements.MaxLeafSize',NewModel)
#bring cultivar descriptions from master back into replacement wheat model
NewModel = findModel(Released,'Wheat.Cultivars')
replaceModel(Master,'Replacements.Wheat.Cultivars',NewModel)
#rename manager scripts to capture max leaf size
renameModelofType(Master,'MaxLeafSize',"Models.Manager, Models",'OutputMaxLeafSize')

In [19]:
os.remove(ImplementedFile)
with open(ImplementedFile,'w') as ImplementedJSON:
    json.dump(Master ,ImplementedJSON,indent=2)

In [20]:
replacements = pd.read_excel(VariableRenamesFile,index_col=0,sheet_name = 'SimpleLeafRenames').to_dict()['SimpleLeaf']
with open(ImplementedFile, 'r') as file: 
    data = file.read() 
    for v in replacements.keys():
        data = data.replace(v, replacements[v])
        w = v.replace('Wheat','[Wheat]')
        rw = replacements[v].replace('Wheat','[Wheat]')
        data = data.replace(w, rw)
        
# Opening our text file in write only 
# mode to write the replaced content 
with open(ImplementedFile, 'w') as file: 
  
    # Writing the replaced data in our 
    # text file 
    file.write(data) 

In [21]:
VariableRenames = pd.read_excel(VariableRenamesFile,index_col=0, sheet_name='SimpleLeafRenames').to_dict()['SimpleLeaf']
MaxLeafSizeRenames = pd.read_excel(VariableRenamesFile,index_col=0, sheet_name='MaxLeafSizeRenames').to_dict()['SimpleLeaf']

from pathlib import Path
fileLoc = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\data'
Allcols = []
pathlist = Path(fileLoc).glob('**/*.xlsx')
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='Observed')
    newCols = []
    replace = False
    for c in obsDat.columns:
        if c in VariableRenames.keys():
            newCols.append(c.replace(c,VariableRenames[c]))
            replace = True
            if c == "Wheat.Leaf.Tips":
                print(str(path) + " tips")
        else:
            newCols.append(c)
    if replace == True:
        obsDat.columns = newCols
        with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
            workbook = writer.book
            obsDat.to_excel(writer,index=False,sheet_name='Observed')
    
    try:
        obsDat = pd.read_excel(path, engine='openpyxl',sheet_name='MaxLeafSize')
        newCols = []
        replace = False
        for c in obsDat.columns:
            if c in MaxLeafSizeRenames.keys():
                newCols.append(c.replace(c,MaxLeafSizeRenames[c]))
                replace = True
            else:
                newCols.append(c)
        if replace == True:
            obsDat.columns = newCols
            with pd.ExcelWriter(path, engine='openpyxl', mode='a',if_sheet_exists='replace') as writer: 
                workbook = writer.book
                obsDat.to_excel(writer,index=False,sheet_name='MaxLeafSize')
    except:
        do = "Nothing"

C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\data\Observed.xlsx tips


In [30]:
from pathlib import Path
fileLoc = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\inputs'
Allcols = []
pathlist = Path(fileLoc).glob('**/*.csv')

In [34]:
VariableRenames

{'Wheat.Grain.Moisture': 'Wheat.Grain.WaterContent',
 'Wheat.Leaf.NonStructural.Wt': 'Wheat.Leaf.Live.StorageWt',
 'Wheat.Phenology.AnthesisDAS': 'Wheat.Phenology.FloweringDAS',
 'Wheat.Phenology.HaunStageTerminalSpikelet': 'Wheat.Phenology.CAMP.TSHS',
 'Wheat.Phenology.ReadyForHarvestDAS': 'Wheat.Phenology.MaturityDAS',
 'Wheat.Spike.NonStructural.Wt': 'Wheat.Spike.Live.StorageWt',
 'Wheat.Stem.NonStructural.Wt': 'Wheat.Stem.Live.StorageWt',
 'Wheat.Structure.HaunStageFloralInitiation': 'Wheat.Phenology.CAMP.VSHS',
 'Wheat.SowingData.Population': 'Wheat.Population',
 'Wheat.SowingData.Population.se': 'Wheat.Population.se',
 'Wheat.Arbitrator.DM.TotalFixationSupply': 'Wheat.Arbitrator.Carbon.TotalFixationSupply',
 'Wheat.Arbitrator.DM.TotalPlantDemand': 'Wheat.Arbitrator.Carbon.TotalPlantDemand',
 'Wheat.Grain.CriticalNConc': 'Wheat.Grain.Nitrogen.Concentration.Critical',
 'Wheat.Grain.DMDemand.Total': 'Wheat.Grain.Carbon.Demands.Total',
 'Wheat.Grain.Live.StructuralN': 'Wheat.Grain.Li

In [36]:
VariableRenames = pd.read_excel(VariableRenamesFile,index_col=0, sheet_name='SimpleLeafRenames').to_dict()['SimpleLeaf']
from pathlib import Path
fileLoc = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\inputs'
Allcols = []
pathlist = Path(fileLoc).glob('**/*.csv')
for path in pathlist:
    # because path is object not string
    obsDat = pd.read_csv(path)
    newCols = []
    replace = False
    for c in obsDat.columns:
        if c in VariableRenames.keys():
            newCols.append(c.replace(c,VariableRenames[c]))
            replace = True
        else:
            newCols.append(c)
    if replace == True:
        print(newCols)
        obsDat.columns = newCols
        obsDat.to_csv(path,index=False)

['SimulationName', '[Wheat].Phenology.CAMP.FLNparams.MinLN', '[Wheat].Leaf.Phyllochron.BasePhyllochron.FixedValue', '[Wheat].Leaf.PhyllochronPpSensitivity.FixedValue', '[Wheat].Phenology.HeadEmergenceLongDayBase.FixedValue', '[Wheat].Phenology.HeadEmergencePpSensitivity.FixedValue', '[Wheat].Phenology.CAMP.FLNparams.PpLN', '[Wheat].Phenology.CAMP.FLNparams.VrnLN', '[Wheat].Phenology.CAMP.FLNparams.VxPLN', '[Wheat].Phenology.Flowering.Target.FixedValue']


In [22]:
## Find and delete model of name and type
# def removeModel(Parent,modelName,modelType):
#     pos = 0
#     for c in Parent['Children']:
#         if (c['Name'] == modelName) and (c['$type'] == modelType):
#             del Parent['Children'][pos]
#             found = True
#             break
#         else:
#             removeModel(c,modelName,modelType)
#         pos += 1
        
# MasterFile = 'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.apsimx'
# with open(MasterFile,'r') as MasterJSON:
#     Master = json.load(MasterJSON)
#     MasterJSON.close()
        
# removeModel(Master,"SetCropParams","Models.Manager, Models")
# removeModel(Master,"SowingReport","Models.Report, Models")

# with open(MasterFile,'w') as MasterJSON:
#     json.dump(Master ,MasterJSON,indent=2)